# 05 · Property models (B): one parameter from others measured on the same sample

Transformation models (Phoon & Kulhawy 1999): e.g. void ratio from water content. Inputs: the parameters in `settings.toml → [property] inputs` (default: water content), depth and the logged soil class. **No location.** These models fill in parameters that were not tested where water content was, for the settlement calculation.

Same validation as notebook 02: V3's 106 m blockCV folds (whole boreholes are tested), nested Optuna (30 trials). References: **Linear** (least squares on the same inputs, the classical correlation) and **Depth** (depth only). Model choice as in notebook 03.

**Run time (estimate):** a few minutes per target (RF, XGB only).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
from tqdm import tqdm

H.set_features(H.property_features(S))
print("Inputs:", H.FEATURES)
runs = H.run_list(S, "property")
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["cv"]["k"] + 1))
main_block = H.block_sizes(S)[0]
PRIV, O = H.out_dir(S, "05_property", private=True), H.out_dir(S, "05_property")
(PRIV / "checkpoints").mkdir(exist_ok=True)
dev = {t: pd.read_csv(H.processed_dir(S) / f"property_{t}.csv").query("set == 'development'") for t in S["property"]["targets"]}
display(pd.DataFrame(runs))

## Run (or continue)

In [ ]:
for r in runs:
    per_fold = max(H.n_trials(r["model"], S), 1)
    ck = lambda k: PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib"
    done = {k: H.scores(*joblib.load(ck(k))["predictions"][["y", "y_pred"]].values.T) for k in outer if ck(k).exists()}
    bar = tqdm(total=len(outer) * per_fold, initial=len(done) * per_fold, desc=f"{r['tag']:<34}", mininterval=10,
               bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]{postfix}")
    for k in outer:
        if k in done:
            continue
        res = H.run_fold(r, dev[r["target"]], k, S, on_trial=lambda: bar.update(1))
        if r["model"] in H.NO_TUNING:
            bar.update(1)
        res.pop("model")
        joblib.dump(res, ck(k), compress=3)
        done[k] = H.scores(res["predictions"].y, res["predictions"].y_pred)
    bar.set_postfix_str(f"mean RMSE {np.nanmean([v['rmse'] for v in done.values()]):.3f} R2 {np.nanmean([v['r2'] for v in done.values()]):.3f}")
    bar.close()

## Results

In [ ]:
preds = pd.concat([joblib.load(f)["predictions"] for f in sorted((PRIV / "checkpoints").glob("*.joblib"))], ignore_index=True)
preds.to_csv(PRIV / "outer_predictions.csv", index=False)
folds = H.fold_scores(preds); folds.to_csv(O / "fold_scores.csv", index=False)
summ = H.summary(folds); summ.to_csv(O / "summary.csv", index=False)
pooled = H.score_table(preds, H.RUN_KEYS); pooled.to_csv(O / "pooled_scores.csv", index=False)
cols = ["target", "model", "rmse_mean", "rmse_sd", "mae_mean", "r2_mean", "spearman_mean", "skill_vs_depth"]
display(summ[cols].round(3))
display(pooled[["target", "model", "n", "rmse", "r2"]].round(3).rename(columns={"r2": "pooled_r2"}))
choice = pd.DataFrame([H.select_model(summ, t, main_block) for t in summ.target.unique()])
choice.to_csv(O / "model_choice.csv", index=False)
display(choice)

In [ ]:
ts = list(summ.target.unique()); models = list(preds.model.unique())
fig, axes = plt.subplots(len(ts), len(models), figsize=(3 * len(models), 3 * len(ts)), squeeze=False)
for i, t in enumerate(ts):
    for j, mod in enumerate(models):
        g = preds[(preds.target == t) & (preds.model == mod)]
        ax = axes[i, j]
        ax.scatter(g.y, g.y_pred, s=4, alpha=0.3)
        lim = [min(g.y.min(), g.y_pred.min()), max(g.y.max(), g.y_pred.max())]
        ax.plot(lim, lim, "k--", lw=0.8)
        ax.set_title(f"{t} · {mod}", fontsize=9); ax.set_xlabel("observed"); ax.set_ylabel("predicted")
plt.tight_layout(); plt.savefig(O / "observed_vs_predicted.png", dpi=200); plt.show()
print(f"Finished in {(time.time() - T0) / 60:.1f} min.")